In [ ]:
import sys
import os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)



In [ ]:
import torch
import random
import numpy as np
import csv

In [ ]:
from models.semcovert import create_network
from utils.video_utils import load_video_as_tensor,split_video_tensor, get_video_files

In [ ]:
def seet_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seet_seed(42)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.ticker import FormatStrFormatter

def plot_custom_line_chart(data_groups, 
                           x_labels, 
                           y_label, 
                           legend_labels,
                           title="Custom Line Chart",
                           str_format='%.2f',
                           colors=None,
                           markers=None,
                           linestyles=None,
                           save_path=None):
    
    plt.figure(figsize=(16, 10))
    
    default_colors = ['#0072B2', '#E69F00', '#882255', '#009E73',
                               '#CC79A7', '#D55E00', '#56B4E9', '#F0E442',
                               '#999999', '#FF69B4', '#8B4513', '#2F4F4F',]

    default_markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
    default_linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (3, 5, 1, 5))]
    
    legend_handles = []  
    
    for i, (x_data, y_data) in enumerate(data_groups):
        color = colors[i] if colors and i < len(colors) else default_colors[i]
        marker = markers[i] if markers and i < len(markers) else default_markers[i % len(default_markers)]
        linestyle = linestyles[i] if linestyles and i < len(linestyles) else default_linestyles[i % len(default_linestyles)]

        plt.plot(x_data, y_data, 
                 linestyle=linestyle, 
                 linewidth=3.0,
                 color=color,
                 alpha=0.85,
                 zorder=1)
        
        plt.plot(x_data, y_data, 
                 marker=marker,
                 linestyle='none',  
                 markersize=14,
                 markeredgewidth=3.0,
                 markeredgecolor=color,
                 markerfacecolor='white',  
                 color=color,
                 zorder=10)

        custom_handle = Line2D([0], [0],
                               linestyle=linestyle,
                               linewidth=2.5,
                               color=color,
                               marker=marker,
                               markersize=10,
                               markeredgewidth=2.5,
                               markeredgecolor=color,
                               markerfacecolor='white')
        legend_handles.append(custom_handle)
    
    plt.xlabel(x_labels[0], fontsize=32, fontweight='normal')
    plt.ylabel(y_label, fontsize=32, fontweight='normal')
    
    # plt.title(title, fontsize=16, fontname='Times New Roman')

    plt.legend(handles=legend_handles, labels=legend_labels, loc='best', fontsize=30, framealpha=0.7, edgecolor='none')
    
    plt.xticks(fontsize=32)
    plt.yticks(fontsize=32)
    
    ax = plt.gca()
    ax.grid(True, which='major', linestyle='-', linewidth=1.2, alpha=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.8, alpha=0.4)
    
    ax.xaxis.set_major_locator(MultipleLocator(2))  
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))  
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

    ax.yaxis.set_major_formatter(FormatStrFormatter(str_format))  # Format y-axis labels
    
    ax.set_facecolor("#ffffff")
    for spine in ax.spines.values():
        spine.set_linewidth(2)  
        spine.set_color("#000000")
    
    plt.tight_layout()
    
    if save_path:
        if not os.path.exists(os.path.dirname(save_path)):
            os.makedirs(os.path.dirname(save_path))
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Fig Saved: {save_path}")
    
    plt.show()

In [ ]:
def load_videos(data_dir, target_size=(240,320), max_frames=5): 
    
    video_files = get_video_files(data_dir)
    if not video_files:
        raise ValueError(f"No video files found in {data_dir}")
    
    print(f"Found {len(video_files)} video files: {[f.name for f in video_files]}")
    
    all_video_batches = []
    for video_file in video_files:
        video_tensor = load_video_as_tensor(
            str(video_file), 
            target_size=target_size
        )
        video_tensor = split_video_tensor(video_tensor, max_frames)
        if video_tensor.numel() > 0:
            for i in range(video_tensor.shape[0]):
                all_video_batches.append(video_tensor[i])
            del video_tensor  
        else:
            print(f"  Failed to load video: {video_file}")
    
    if not all_video_batches:
        raise ValueError("No valid video data loaded")
    
    all_video_tensor = torch.stack(all_video_batches, dim=0)

    return all_video_tensor

In [ ]:
def generate(model, cover_video, batch_size, device):
    recon_secret = torch.empty_like(cover_video)
    with torch.no_grad():
        for i in range(0, cover_video.shape[0], batch_size):
            batch_cover = cover_video[i:i + batch_size].to(device)
            encode_results = model.encode_videos(batch_cover)
            z_cover = encode_results['z_cover']
            z_received = model.transmit_through_channel(z_cover)
            z_secret_extracted = model.extract_secret(z_received)
            recon_secret_batch = model.vae.decode(z_secret_extracted)

            recon_secret[i:i + batch_size] = recon_secret_batch.cpu()

    return recon_secret

In [ ]:
model_cfg = {
        'depth': 4,
        'dim': 96,
        'use_channel': True,
        'vae_config': {
            'dim': 96,
            'z_dim': 16,
            'dim_mult': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attn_scales': [],
            'temperal_downsample': [False, True, True],
            'dropout': 0.0
        },
        'channel_config': {
            'channel_type': 'AWGN',  # Channel type
            'snr': 25,               # Signal-to-noise ratio
        },    
    }

In [ ]:
def run(model, data_dir, batch_size=8, max_batch=1000, target_size=(240, 320), device='cuda'):
    cover_videos = load_videos(data_dir, target_size=target_size, max_frames=5)

    cover_videos = cover_videos[:max_batch] if cover_videos.shape[0] > max_batch else cover_videos

    diff_snr = [5,9,13,17,21,25,29,33]
    mse_list = []
    mse_std_list = []

    for snr in diff_snr:
        model.set_channel(
            {
                'channel_type': "AWGN",
                'snr': snr
            }
        )
        recon_secret = generate(model, cover_videos, batch_size, device)
        zero_tensor = torch.zeros_like(recon_secret)
        mse = torch.mean((recon_secret - zero_tensor) ** 2, dim=(1, 2, 3))
        mse_std = torch.std(mse)
        mse_mean = torch.mean(mse)

        mse_list.append(mse_mean.item())
        mse_std_list.append(mse_std.item())

        print(f"SNR: {snr} dB, MSE: {mse_mean.item():.7f}, Std: {mse_std.item():.7f}")

    return mse_list, mse_std_list


In [ ]:

# Model for AWGN channel
model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)

data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for UCF101 Dataset

batch_size = 16
max_batch = 1000
mse_listA, mse_std_listA = run(model, data_dir, batch_size, max_batch, (240,320), device)

data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for DAVIS Dataset
batch_size = 4
max_batch = 500
mse_listB, mse_std_listB = run(model, data_dir, batch_size, max_batch, (480,640), device)

data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for MOT17 Dataset
batch_size = 1
max_batch = 100
mse_listC, mse_std_listC = run(model, data_dir, batch_size, max_batch, (1080,1920), device)

diff_snr = [5,9,13,17,21,25,29,33]

data_groups = [
    (diff_snr, mse_listA),
    (diff_snr, mse_listB),
    (diff_snr, mse_listC)
]

save_path = "YOUR_SAVE_PATH"  # Path to save the figures
plot_custom_line_chart(data_groups, 
                        x_labels=['SNR (dB)'], 
                        y_label='MSE',
                        legend_labels=['UCF101', 'DAVIS','MOT17'],
                        save_path=str(save_path)+'/mse_comparison.pdf')

# Save the results to a CSV file
csv_file = os.path.join(save_path, 'mse_results.csv')
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['SNR (dB)', 'UCF101 MSE', 'DAVIS MSE', 'MOT17 MSE'])
    for i, snr in enumerate(diff_snr):
        writer.writerow([snr, mse_listA[i], mse_listB[i], mse_listC[i]])
